In [2]:
import numpy as np
import cupy as cp
import itertools
from scipy.spatial.transform import Rotation

In [3]:
# E^2 shift for my parameters for tau=1e-3, T=0.3K, E=100 kV/cm: 4e-5 rad/s
# B^2 shift for my parameters for tau=1e-3, T=0.3K, dBx/dx = 0.1 part per meter : 0.0711 rad/s

In [757]:
neutrongyro = -1.83247172e8 # rad/Tesla
he3gyro = -2.037894585e8
kB = 1.38e-23 #SI
m3 = 1.15e-26

T = 0.3 # Kelvin
B0 = 5.2e-6 # Tesla
Bg = 0 # ppcm
Ez = 1e8 # V/cm
d = 1e-2 # Lattice spacing, m
dt = d/np.sqrt(3*kB*T/m3) # Time step, s
Nx = 41    # resolution x-dir
Ny = 11    # resolution y-dir
Nz = 8    # resolution z-dir
tau = 0.000592/dt + 0.5   # collision timescale in units of dt. 
#TODO Not sure about this 1/2 renormalization...
Nt = 100   # number of timesteps

if tau < 1:
    print("Warning: short relaxation times can lead to numerical instability")

In [748]:
def D3Q15():
    NL = 15 # Lattice speeds / weights
    vs = np.array([[0, 0, 0], #0
                   [1, 0, 0], #1
                   [-1, 0, 0], #2
                   [0, 1, 0], #3
                   [0, -1, 0], #4
                   [0, 0, 1], #5
                   [0, 0, -1], #6
                   [1, 1, 1], #7
                   [1, 1, -1], #8
                   [1, -1, 1], #9
                   [1, -1, -1], #10
                   [-1, 1, 1], #11
                   [-1, 1, -1], #12
                   [-1, -1, 1], #13
                   [-1, -1, -1]]) #14

    weights = cp.array([2/9,
                        1/9, 1/9, 1/9, 1/9, 1/9, 1/9,
                        1/72, 1/72, 1/72, 1/72, 1/72, 1/72, 1/72, 1/72]) #Lattice weights

    reverse_x = cp.array([0,2,1,3,4,5,6,11,12,13,14,7,8,9,10])
    reverse_y = cp.array([0,1,2,4,3,5,6,9,10,7,8,13,14,11,12])
    reverse_z = cp.array([0,1,2,3,4,6,5,8,7,10,9,12,11,14,13])
    return NL, vs, weights, reverse_x, reverse_y, reverse_z

def D3Q19():
    NL = 19 # Lattice speeds / weights
    vs = np.array([[0, 0, 0], #0
                   [1, 0, 0], #1
                   [-1, 0, 0], #2
                   [0, 1, 0], #3
                   [0, -1, 0], #4
                   [0, 0, 1], #5
                   [0, 0, -1], #6
                   [1, 1, 0], #7
                   [1, -1, 0], #8
                   [-1, 1, 0], #9
                   [-1, -1, 0], #10
                   [1, 0, 1], #11
                   [1, 0, -1], #12
                   [-1, 0, 1], #13
                   [-1, 0, -1], #14
                   [0, 1, 1], #15
                   [0, 1, -1], #16
                   [0, -1, 1], #17
                   [0, -1, -1] #18
                  ])

    weights = cp.array([1/3] + [1/18] * 6 + [1/36] * 12) #Lattice weights
    reverse_x = cp.array([0,2,1,3,4,5,6,9,10,7,8,13,14,11,12,15,16,17,18])
    reverse_y = cp.array([0,1,2,4,3,5,6,8,7,10,9,11,12,13,14,17,18,15,16])
    reverse_z = cp.array([0,1,2,3,4,6,5,7,8,9,10,12,11,14,13,16,15,18,17])
    return NL, vs, weights, reverse_x, reverse_y, reverse_z

In [749]:
def check_reversal(vs, reverse_x, reverse_y, reverse_z):
    assert np.all(vs[reverse_x.get(),:] == vs @ np.array([[-1, 0, 0], [0, 1, 0], [0, 0, 1]]))
    assert np.all(vs[reverse_y.get(),:] == vs @ np.array([[1, 0, 0], [0, -1, 0], [0, 0, 1]]))
    assert np.all(vs[reverse_z.get(),:] == vs @ np.array([[1, 0, 0], [0, 1, 0], [0, 0, -1]]))

In [758]:
NL, vs, weights, reverse_x, reverse_y, reverse_z = D3Q19()
check_reversal(vs, reverse_x, reverse_y, reverse_z)
# Initial Conditions
M = cp.zeros((NL,3,Nx,Ny,Nz))
M[:,0,:,:,:] = cp.tile(cp.reshape(weights, (NL,1,1,1)), (1,Nx,Ny,Nz))

kTm = ((d/dt)**2)/3
print("Effective temperature: %.4f K" % (kTm * m3/kB))

Effective temperature: 0.3000 K


In [751]:
def get_B(v,i,j,k):
    Bholding = np.array([0, 0, B0])
    # Pure dBx/dx gradient. Maxwell? Who's that?
    Bgrad = np.array([Bg * B0 * (i - Nx/2), 0, 0])
    # E field
    Eholding = np.array([0, 0, Ez])
    BvE = np.cross(vs[v,:] * d/dt, Eholding)/(3e8)**2 
    return Bholding + Bgrad + BvE

def compute_rotation(B, dt):
    return Rotation.from_rotvec(B * he3gyro * dt).as_matrix()

U = np.zeros((3,3,NL,Nx,Ny,Nz))
for v,i,j,k in itertools.product(range(NL), range(Nx), range(Ny), range(Nz)):
    B = get_B(v,i,j,k)
    U[:,:,v,i,j,k] = compute_rotation(B,dt)
U = cp.array(U);

In [759]:
for it in range(Nt):
    # Drift
    for i in range(NL):
        M[i,:,:,:,:] = cp.roll(M[i,:,:,:,:], (vs[i,0], vs[i,1], vs[i,2]), axis=(1,2,3))

    # Apply boundary
    M[:,:,0,:,:] = M[reverse_x,:,0,:,:]
    M[:,:,:,0,:] = M[reverse_y,:,:,0,:]
    M[:,:,:,:,0] = M[reverse_z,:,:,:,0]
    M[:,:,Nx-1,:,:] = M[reverse_x,:,Nx-1,:,:]
    M[:,:,:,Ny-1,:] = M[reverse_y,:,:,Ny-1,:]
    M[:,:,:,:,Nz-1] = M[reverse_z,:,:,:,Nz-1]
    
    # Apply Collision
    Minterior = M[:,:,1:Nx-1,1:Ny-1,1:Nz-1]
    M[:,:,1:Nx-1,1:Ny-1,1:Nz-1] += (cp.multiply.outer(weights,cp.sum(Minterior,axis=0)) - Minterior)/tau
    #M += (cp.multiply.outer(weights,cp.sum(M,axis=0)) - M)/tau
    
    #Bloch Equations
    M = cp.einsum('bavijk,vbijk->vaijk', U, M)
    

In [760]:
MFinal = cp.sum(M, axis=(0,2,3,4))

In [761]:
MFinal = cp.sum(M[:,:,1:Nx-1,1:Ny-1,1:Nz-1], axis=(0,2,3,4))

In [762]:
# Evaluate phase shift
MFinal

array([ 1.42100032e+03,  1.55364926e+03, -4.87890978e-19])

In [763]:
# Evaluate phase shift
(np.arctan2(MFinal[1], MFinal[0])  - ((-he3gyro * B0 * dt * Nt + np.pi) % (2 * np.pi) - np.pi))/(dt * Nt)

array(0.00284063)